In [1]:
import sys, pickle
from pathlib import Path
import pandas as pd

# 1. Repo Discovery (Boilerplate to find the root 'syllable_generation' folder)
# ---------------------------------------------------------------------
# Assuming notebook is in: .../syllable_generation/projects/jonathan_openfield_example/
current_dir = Path.cwd() 
repo_root = current_dir.parents[1] # Go up 2 levels to find repo root
if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

# 2. Load the "Dirty" Sessions
# ---------------------------------------------------------------------
# Adjust this path if you saved the pickle somewhere else!
# Assuming you saved it in the standard 'loaders' folder:
DIRTY_PICKLE_PATH = repo_root / "loaders" / "jonathan_amph_sessions.pkl"

if not DIRTY_PICKLE_PATH.exists():
    raise FileNotFoundError(f"Could not find your dirty pickle at: {DIRTY_PICKLE_PATH}")

print(f"Loading dirty sessions from: {DIRTY_PICKLE_PATH}")
with open(DIRTY_PICKLE_PATH, "rb") as f:
    sessions = pickle.load(f)

print(f"Loaded {len(sessions)} sessions.")

# 3. Define Output Path for exclusions.csv
# ---------------------------------------------------------------------
# Since we are in 'projects/jonathan_openfield_example', overrides is right here.
OUTPUT_CSV = current_dir / "overrides" / "exclusions.csv"

# Ensure the folder exists
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
print(f"Exclusions will be saved to: {OUTPUT_CSV}")

Loading dirty sessions from: /home/fsp585/Desktop/FiberphotometryCode/syllable_generation/loaders/jonathan_amph_sessions.pkl


/tmp/ipykernel_704893/979277828.py:24: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  sessions = pickle.load(f)


Loaded 30 sessions.
Exclusions will be saved to: /home/fsp585/Desktop/FiberphotometryCode/syllable_generation/projects/jonathan_openfield_amph/overrides/exclusions.csv


In [ ]:
import ipywidgets as widgets
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# --- Aux Function ---
def map_sessions_to_npz(sessions, npz_data):
    mapping = {}
    npz_keys = list(npz_data.files)
    for s in sessions:
        match = next((k for k in npz_keys if s.session_id in k), None)
        if match: mapping[s.session_id] = match
    return mapping

# --- Main Class (Updated for SessionID Support) ---
class PhotometryCleaner:
    def __init__(self, sessions, output_path, npz_path=None):
        self.sessions = sorted(sessions, key=lambda s: s.session_id)
        # Create a quick lookup for metadata (SessionID -> SessionObj)
        self.session_dict = {s.session_id: s for s in sessions}
        
        self.output_path = Path(output_path).resolve()
        self.npz_path = Path(npz_path).resolve() if npz_path else None
        self.symlink_path = self.output_path.parent / "auto_mask.npz"
        self.reviewed_path = self.output_path.parent / "reviewed_sessions.txt"
        
        # KEY CHANGE: Exclusions are now stored by SessionID, not AnimalID
        # Format: { 'Session_123': { 'DLS-R': [(start, end), ...] } }
        self.exclusions = {} 
        self.npz_data = None
        self.session_to_npz_key = {}
        self.reviewed = set()
        
        # Load Data
        self._load_existing_csv()
        self._load_reviewed()
        self._init_npz()
        
        # State
        self.current_sess_idx = 0
        self.current_signals = {} 
        self.current_sig_key = None 
        
        # --- UI Elements ---
        self.out = widgets.Output()
        
        self.drop_session = widgets.Dropdown(
            options=self._get_session_options(), 
            value=0, 
            description='Session:', 
            layout=widgets.Layout(width='400px')
        )
        self.drop_signal = widgets.Dropdown(description='Signal:', layout=widgets.Layout(width='200px'))
        
        self.chk_reviewed = widgets.Checkbox(value=False, description='Mark Reviewed', indent=False, layout=widgets.Layout(width='120px'))
        self.prog_label = widgets.Label(value=f"({len(self.reviewed)}/{len(self.sessions)})")

        self.btn_prev_sess = widgets.Button(description="< Sess", layout=widgets.Layout(width='70px'))
        self.btn_next_sess = widgets.Button(description="Sess >", layout=widgets.Layout(width='70px'))
        self.btn_next_sig = widgets.Button(description="Next Signal >", icon='arrow-right')

        self.btn_mode_pan = widgets.Button(description="Pan/Zoom", icon='arrows', button_style='')
        self.btn_mode_select = widgets.Button(description="Box Select", icon='crosshairs', button_style='info')

        self.btn_save = widgets.Button(description="Save Config", button_style='success', icon='save')
        self.btn_undo = widgets.Button(description="Undo Last", button_style='warning', icon='undo')
        
        mode_text = "<b>Auto-Mask: ON</b> (Grey traces visible)" if self.npz_data else "<b>Auto-Mask: OFF</b> (Manual mode)"
        self.info_label = widgets.HTML(value=f"<i>{mode_text}</i>")

        # --- Bindings ---
        self.drop_session.observe(self._on_sess_change, names='value')
        self.drop_signal.observe(self._on_sig_change, names='value')
        self.chk_reviewed.observe(self._on_reviewed_change, names='value')
        
        self.btn_prev_sess.on_click(lambda b: self._nav_sess(-1))
        self.btn_next_sess.on_click(lambda b: self._nav_sess(1))
        self.btn_next_sig.on_click(self._nav_sig)
        
        self.btn_mode_pan.on_click(self._set_mode_pan)
        self.btn_mode_select.on_click(self._set_mode_select)
        
        self.btn_save.on_click(self._save_config)
        self.btn_undo.on_click(self._undo_last)

        # --- Layout ---
        self.layout = widgets.VBox([
            widgets.HBox([self.btn_prev_sess, self.drop_session, self.btn_next_sess, self.chk_reviewed, self.prog_label]),
            widgets.HBox([self.drop_signal, self.btn_next_sig, widgets.Label(" | Tools: "), self.btn_mode_pan, self.btn_mode_select]),
            self.out,
            widgets.HBox([self.btn_save, self.btn_undo, self.info_label])
        ])
        
        self._load_session_data(0)

    def _get_session_options(self):
        opts = []
        for i, s in enumerate(self.sessions):
            prefix = "✔️ " if s.session_id in self.reviewed else ""
            label = f"{prefix}{s.session_id}"
            opts.append((label, i))
        return opts

    def _init_npz(self):
        if not self.npz_path or not self.npz_path.exists(): return
        try:
            raw_npz = np.load(self.npz_path)
            
            # --- METADATA DRIVEN SCALING ---
            # Try to find model_summary.json in the same folder as the NPZ (assuming typical structure)
            parent = self.npz_path.parent
            summary_path = parent / "model_summary.json"
            rate = 2.0 # Default fallback (High-Res Anchor)
            
            # 1. Try explicit summary
            if summary_path.exists():
                try:
                    import json
                    with open(summary_path, 'r') as f:
                        meta = json.load(f)
                        if "downsample_rate" in meta:
                            rate = float(meta["downsample_rate"])
                            print(f"Found downsample_rate in summary: {rate}")
                except: pass
            
            # 2. Try config.yml fallback check
            # (Simplified check: if rate is default 2.0, check config to see if it should be 1.0)
            else:
                 config_path = parent.parent / "config.yml" # Usually one level up (amph_project/config.yml)
                 if config_path.exists():
                    try:
                         import yaml
                         with open(config_path, 'r') as f:
                             cfg = yaml.safe_load(f)
                             fps = float(cfg.get("fps", 40))
                             if fps > 50: # If model is 60Hz or 80Hz
                                 rate = 1.0
                                 print(f"Config FPS {fps} implies rate 1.0")
                    except: pass
            print(f"Applying scaling factor: x{rate}")
            # Scale the indices by the rate
            self.npz_data = {k: np.round(raw_npz[k] * rate).astype(int) for k in raw_npz.files}
            
            # Map sessions
            self.session_to_npz_key = map_sessions_to_npz(self.sessions, raw_npz) 
            
        except Exception as e: 
            print(f"Error loading NPZ: {e}")
            import traceback
            traceback.print_exc()

    def _load_existing_csv(self):
        if self.output_path.exists():
            try:
                df = pd.read_csv(self.output_path, dtype=str).fillna("")
                for _, row in df.iterrows():
                    # Support both new "SessionID" and old "ID" columns
                    sid = str(row.get("SessionID", "")).strip()
                    if not sid: sid = str(row.get("Session", "")).strip()
                    
                    aid = str(row.get("ID", "")).strip()
                    reg = row.get("Region")
                    intervals = row.get("Intervals", "")

                    if not reg or not intervals: continue
                    
                    # Logic: If SessionID exists, apply to that session.
                    # If NOT (legacy CSV), apply to ALL sessions with that Animal ID.
                    targets = []
                    if sid:
                        targets.append(sid)
                    elif aid:
                        # Legacy fallback: find all sessions for this animal
                        targets = [s.session_id for s in self.sessions 
                                   if str(s.metadata.get('animal_id', '')).strip() == aid]
                    
                    for target_sid in targets:
                        if target_sid not in self.exclusions: self.exclusions[target_sid] = {}
                        if reg not in self.exclusions[target_sid]: self.exclusions[target_sid][reg] = []
                        
                        for part in intervals.split(";"):
                            if "idx:" in part:
                                try:
                                    rng = part.replace("idx:", "").strip().split("-")
                                    self.exclusions[target_sid][reg].append((float(rng[0]), float(rng[1])))
                                except: pass
            except Exception as e: print(f"CSV Load Error: {e}")

    def _load_reviewed(self):
        if self.reviewed_path.exists():
            try:
                with open(self.reviewed_path, 'r') as f:
                    self.reviewed = set(line.strip() for line in f if line.strip())
            except: pass

    def _get_signals_data(self, session):
        # ... (Same as before, simplified for brevity in this answer) ...
        # (Copying the logic from previous version verbatim as it didn't need changes)
        signals = {}
        sig_meta = session.metadata.get('signal_meta', {})
        phot = getattr(session, 'photometry', {})
        
        valid_set = None
        if self.npz_data and session.session_id in self.session_to_npz_key:
            npz_key = self.session_to_npz_key[session.session_id]
            valid_set = set(self.npz_data[npz_key])
            
        for sig_col, meta in sig_meta.items():
            orig_letter = meta.get('orig_letter', 'G').upper()
            target_key = '470' if orig_letter == 'G' else ('560' if orig_letter == 'R' else '415')
            
            if phot and target_key in phot:
                df = phot[target_key]
                reg = meta.get('brain_reg', 'Unknown')
                hemi = meta.get('hemisphere', '')
                lbl = f"{reg}-{hemi}" if hemi else reg
                unique = f"{sig_col}_{orig_letter}"
                
                raw_y = df[sig_col].values
                frames = df['frame_idx'].values
                
                masked_y = raw_y.copy()
                if valid_set is not None:
                    mask = np.isin(frames, list(valid_set))
                    masked_y[~mask] = np.nan
                
                y_min, y_max = np.nanmin(masked_y), np.nanmax(masked_y)
                if np.isnan(y_min): y_min, y_max = np.nanmin(raw_y), np.nanmax(raw_y)
                if np.isnan(y_min): y_min, y_max = 0, 1
                
                yrange = y_max - y_min
                y_min -= yrange * 0.05
                y_max += yrange * 0.05
                
                signals[unique] = {
                    'x': frames, 
                    'y_raw': raw_y,      
                    'y_masked': masked_y, 
                    'ylims': [y_min, y_max],
                    'reg_key': lbl, 
                    'display': f"{lbl} ({orig_letter})",
                    'color': '#2ca02c' if orig_letter == 'G' else '#d62728'
                }
        return signals

    def _load_session_data(self, idx):
        self.current_sess_idx = idx
        session = self.sessions[idx]
        self.current_signals = self._get_signals_data(session)
        
        self.chk_reviewed.unobserve(self._on_reviewed_change, names='value')
        self.chk_reviewed.value = session.session_id in self.reviewed
        self.chk_reviewed.observe(self._on_reviewed_change, names='value')
        
        opts = [(v['display'], k) for k, v in self.current_signals.items()]
        
        self.drop_signal.unobserve(self._on_sig_change, names='value')
        self.drop_signal.options = opts
        if opts:
            self.drop_signal.value = opts[0][1]
            self.current_sig_key = opts[0][1]
        else:
            self.drop_signal.value = None
            self.current_sig_key = None
        self.drop_signal.observe(self._on_sig_change, names='value')
        
        self.render_plot()

    def render_plot(self):
        self.out.clear_output(wait=True)
        if not self.current_signals or not self.current_sig_key:
            with self.out: print("No Data Available")
            return

        session = self.sessions[self.current_sess_idx]
        sid = session.session_id  # USE SESSION ID
        aid = session.metadata.get('animal_id', 'Unknown')
        data = self.current_signals[self.current_sig_key]
        
        mode = 'select' if self.btn_mode_select.button_style == 'info' else 'pan'
        
        fig = go.FigureWidget()
        fig.update_layout(
            title=f"<b>{sid}</b> (ID:{aid}) | {data['display']}", 
            dragmode=mode, 
            height=500,
            width=1200, 
            xaxis=dict(fixedrange=False), 
            yaxis=dict(fixedrange=False, range=data['ylims']),
            template="plotly_white",
            margin=dict(l=10, r=10, t=40, b=10),
            showlegend=True
        )
        
        if self.npz_data:
            fig.add_trace(go.Scattergl(
                x=data['x'], y=data['y_raw'], 
                line=dict(color='lightgrey', width=1), opacity=0.5,
                name="Auto-Excluded", mode='lines', hoverinfo='skip'
            ))
        
        fig.add_trace(go.Scattergl(
            x=data['x'], y=data['y_masked'], 
            line=dict(color=data['color'], width=1.5), 
            name="Valid Signal", mode='lines'
        ))

        # Render Exclusions using SESSION ID
        if sid in self.exclusions and data['reg_key'] in self.exclusions[sid]:
            for s, e in self.exclusions[sid][data['reg_key']]:
                fig.add_shape(type="rect", x0=s, x1=e, y0=0, y1=1, xref="x", yref="paper", fillcolor="black", opacity=0.2, line_width=0)

        def on_select(trace, points, selector):
            if not selector or not selector.xrange: return
            s, e = selector.xrange
            # SAVE using SESSION ID
            if sid not in self.exclusions: self.exclusions[sid] = {}
            if data['reg_key'] not in self.exclusions[sid]: self.exclusions[sid][data['reg_key']] = []
            
            self.exclusions[sid][data['reg_key']].append((round(s), round(e)))
            
            with fig.batch_update():
                fig.add_shape(type="rect", x0=s, x1=e, y0=0, y1=1, xref="x", yref="paper", fillcolor="red", opacity=0.4, line_width=0)
            self.info_label.value = f"Excluded: {int(s)}-{int(e)}"

        fig.data[-1].on_selection(on_select)
        self.current_fig = fig
        with self.out: display(fig)

    # --- Handlers ---
    def _nav_sess(self, d):
        new = self.current_sess_idx + d
        if 0 <= new < len(self.sessions): self.drop_session.value = new 
    def _on_sess_change(self, c): 
        if c['new'] != self.current_sess_idx: self._load_session_data(c['new'])
    def _on_sig_change(self, c): 
        self.current_sig_key = c['new']; self.render_plot()
    def _nav_sig(self, b):
        opts = [o[1] for o in self.drop_signal.options]
        try: self.drop_signal.value = opts[(opts.index(self.drop_signal.value) + 1) % len(opts)]
        except: pass
    def _set_mode_pan(self, b):
        self.btn_mode_pan.button_style, self.btn_mode_select.button_style = 'info', ''
        if hasattr(self, 'current_fig'): self.current_fig.update_layout(dragmode='pan')
    def _set_mode_select(self, b):
        self.btn_mode_select.button_style, self.btn_mode_pan.button_style = 'info', ''
        if hasattr(self, 'current_fig'): self.current_fig.update_layout(dragmode='select')
        
    def _undo_last(self, b):
        # Undo using SESSION ID
        sid = self.sessions[self.current_sess_idx].session_id
        if not self.current_sig_key: return
        reg = self.current_signals[self.current_sig_key]['reg_key']
        if sid in self.exclusions and reg in self.exclusions[sid] and self.exclusions[sid][reg]:
            self.exclusions[sid][reg].pop(); self.render_plot()

    def _on_reviewed_change(self, change):
        sid = self.sessions[self.current_sess_idx].session_id
        if change['new']: self.reviewed.add(sid)
        else: self.reviewed.discard(sid)
        self.prog_label.value = f"({len(self.reviewed)}/{len(self.sessions)})"
        current_idx = self.drop_session.value
        self.drop_session.unobserve(self._on_sess_change, names='value')
        self.drop_session.options = self._get_session_options()
        self.drop_session.value = current_idx
        self.drop_session.observe(self._on_sess_change, names='value')

    def _save_config(self, b):
        # 1. Save CSV with SESSION ID
        rows = []
        for sid, regs in self.exclusions.items():
            # Lookup Animal ID for the CSV (for completeness)
            s_obj = self.session_dict.get(sid)
            aid = str(s_obj.metadata.get("animal_id", "")).strip() if s_obj else ""
            
            for r, intervals in regs.items():
                if intervals:
                    s_ints = "; ".join([f"idx:{int(s)}-{int(e)}" for s, e in sorted(intervals)])
                    # WRITE BOTH SessionID AND ID
                    rows.append({"SessionID": sid, "ID": aid, "Region": r, "Intervals": s_ints})
        
        # Save with SessionID as first column
        cols = ["SessionID", "ID", "Region", "Intervals"]
        pd.DataFrame(rows).to_csv(self.output_path, index=False, columns=cols)
        
        # 2. Save Reviewed List
        with open(self.reviewed_path, "w") as f:
            for sid in sorted(self.reviewed):
                f.write(f"{sid}\n")

        # 3. Manage Symlink
        msg_suffix = ""
        if self.npz_path:
            if self.symlink_path.exists() or self.symlink_path.is_symlink():
                self.symlink_path.unlink()
            self.symlink_path.symlink_to(self.npz_path)
            msg_suffix = " + Auto-Mask Linked"
        
        self.info_label.value = f"<b>Saved!</b> ({len(rows)} manual regions, {len(self.reviewed)} reviewed{msg_suffix})"

    def show(self): 
        display(self.layout)

In [ ]:
# 1. Define Paths
NPZ_PATH = Path('/home/fsp585/Desktop/FiberphotometryCode/syllable_generation/syllabizers/keypoint_moseq_syll/amph_project/amph_vfi_test/video_frame_indexes.npz')
CSV_PATH = Path("./overrides/exclusions.csv")
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

# 2. Launch
app = PhotometryCleaner(sessions, CSV_PATH, NPZ_PATH)
app.show()

In [4]:
npz_data = np.load(NPZ_PATH)

In [5]:
for k in npz_data.keys():
    print(k)

Arena1234_CenterTopCamVid_2024-12-16T12_42_27_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-16T14_05_53_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-16T15_14_26_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-16T11_27_19_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-16T12_42_27_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-13T14_50_39_bottom_rightDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-13T13_37_45_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-13T13_37_45_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-1

In [6]:
sessions[0].session_id

'2024-12-13T12_20_06_bottom_left'